In [ ]:
!pip install tensorflow transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [ ]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
from transformers import create_optimizer
from datasets import load_dataset
import numpy as np

In [ ]:
model_checkpoint = "distilbert-base-uncased"
num_labels = 2

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset("imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
def tokenize_function(examples):

    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


tokenized_datasets = dataset.map(tokenize_function, batched=True)


tf_train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=True,
    batch_size=16
)

tf_validation_dataset = tokenized_datasets["test"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["label"],
    shuffle=False,
    batch_size=16
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/arrow_dataset.py:400: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


In [ ]:
num_epochs = 1
batch_size = 16
learning_rate = 5e-5
weight_decay = 0.01
num_train_steps = len(tokenized_datasets["train"]) // batch_size * num_epochs


optimizer, lr_schedule = create_optimizer(
    init_lr=learning_rate,
    num_train_steps=num_train_steps,
    weight_decay_rate=weight_decay,
    num_warmup_steps=0
)


model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=tf.metrics.SparseCategoricalAccuracy()
)

In [ ]:

history = model.fit(
    tf_train_dataset,
    validation_data=tf_validation_dataset,
    epochs=num_epochs
)


1563/1563 [==============================] - 508s 315ms/step - loss: 0.3441 - sparse_categorical_accuracy: 0.8449 - val_loss: 0.2824 - val_sparse_categorical_accuracy: 0.8795


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning/tokenizer_config.json',
 '/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning/special_tokens_map.json',
 '/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning/vocab.txt',
 '/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning/added_tokens.json',
 '/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning/tokenizer.json')

### Prediction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/Luminar Feb 25/Fine-Tuning"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
loaded_model = TFAutoModelForSequenceClassification.from_pretrained(output_dir)
loaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)

Some layers from the model checkpoint at /content/drive/MyDrive/Luminar Feb 25/Fine-Tuning were not used when initializing TFDistilBertForSequenceClassification: ['dropout_19']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Luminar Feb 25/Fine-Tuning and are newly initialized: ['dropout_39']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
text = "the worst movie I ever saw"
inputs = loaded_tokenizer(text, return_tensors = 'tf', padding = True, truncation = True)
outputs = loaded_model(inputs)
predictions = tf.nn.softmax(outputs.logits)
predicted_class = tf.argmax(predictions, axis = 1).numpy()[0]
print(predicted_class)
confidence = predictions[0][predicted_class].numpy()
print(confidence)

0
0.99436545


In [ ]:
def test_loaded_model(text):
    inputs = loaded_tokenizer(text, return_tensors="tf", padding=True, truncation=True)
    outputs = loaded_model(inputs)
    predictions = tf.nn.softmax(outputs.logits, axis=-1)
    predicted_class = tf.argmax(predictions, axis=1).numpy()[0]

    sentiment = "positive" if predicted_class == 1 else "negative"
    confidence = predictions[0][predicted_class].numpy()
    return sentiment, confidence


In [ ]:
test_text = "I thoroughly enjoyed this movie. The acting was superb."
sentiment, confidence = test_loaded_model(test_text)
print(f"Test text: {test_text}")
print(f"Predicted sentiment: {sentiment} (confidence: {confidence:.2f})")

Test text: I thoroughly enjoyed this movie. The acting was superb.
Predicted sentiment: positive (confidence: 0.99)


#### Evaluating the performance of Fine-Tuned model against Pre-Trained model

In [ ]:
model_checkpoint = "distilbert-base-uncased"
num_labels = 2
model = TFAutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
domain_specific_examples =    "The movie is fantastic"
print("Domain-specific examples comparison:")
print("-" * 80)

inputs = loaded_tokenizer(domain_specific_examples, return_tensors="tf", padding=True, truncation=True)

outputs_pretrained = model(inputs)
preds_pretrained = tf.nn.softmax(outputs_pretrained.logits, axis=-1).numpy()[0]
pred_class_pretrained = np.argmax(preds_pretrained)
confidence_pretrained = preds_pretrained[pred_class_pretrained]

outputs_finetuned = loaded_model(inputs)
preds_finetuned = tf.nn.softmax(outputs_finetuned.logits, axis=-1).numpy()[0]
pred_class_finetuned = np.argmax(preds_finetuned)
confidence_finetuned = preds_finetuned[pred_class_finetuned]

print(f"Text: {text}")
print(f"Pre-trained model: Class {pred_class_pretrained} with {confidence_pretrained:.2f} confidence")
print(f"Fine-tuned model: Class {pred_class_finetuned} with {confidence_finetuned:.2f} confidence")
print("-" * 80)

Domain-specific examples comparison:
--------------------------------------------------------------------------------
Text: I thoroughly enjoyed this movie
Pre-trained model: Class 1 with 0.57 confidence
Fine-tuned model: Class 1 with 0.99 confidence
--------------------------------------------------------------------------------
